In [ ]:
%pip install numpy

In [ ]:
import numpy as np

In [ ]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [ ]:
class HiddenMarkovModel:

    def __init__(self, initial, transition, emission):  # constructor

        # self.initial_probs = initial
        # self.transition_probs = transition
        # self.emission_probs = emission

        self.n_states = len(initial)
        self.n_observations = len(list(emission.values())[0])

        states = list(initial.keys())

        self.initial_probs = np.array([initial[state] for state in states])

        # trans prob matrix creation
        self.transition_probs = np.zeros((self.n_states, self.n_states))

        for i, state_1 in enumerate(transition):  # outside loop outer dict
            for j, state_2 in enumerate(transition[state_1]):

                self.transition_probs[i][j] = transition[state_1][state_2]

        self.emission_probs = np.zeros((self.n_states, self.n_observations))
        self.emission_idx_mapping = {}
        for i, state_1 in enumerate(emission):
            for j, state_2 in enumerate(emission[state_1]):

                self.emission_idx_mapping[state_2] = j
                self.emission_probs[i][j] = emission[state_1][state_2]

    def forward(self, O):

        if isinstance(O, str):
            O = [self.emission_idx_mapping[obs] for obs in O]

        T = len(O)
        forward_probs = np.zeros((T, self.n_states))

        # Initialization step
        for i in range(self.n_states):
            forward_probs[0, i] = self.initial_probs[i] * self.emission_probs[i, O[0]]

        # Recursion step
        for t in range(1, T):
            for j in range(self.n_states):
                forward_probs[t, j] = np.sum(forward_probs[t-1] * self.transition_probs[:, j]) * self.emission_probs[j, O[t]]

        return forward_probs

    def backward(self, O):

        if isinstance(O, str):
            O = [self.emission_idx_mapping[obs] for obs in O]

        T = len(O)
        backward_probs = np.zeros((T, self.n_states))

        # Initialization step
        backward_probs[T-1] = np.ones(self.n_states)

        # Recursion step (backwards in time)
        for t in range(T-2, -1, -1):
            for i in range(self.n_states):
                backward_probs[t, i] = np.sum(self.transition_probs[i] * self.emission_probs[:, O[t+1]] * backward_probs[t+1])

        return backward_probs

    def baum_welch(self, obs, n_iterations=10):

        if isinstance(obs, str):
            obs = [self.emission_idx_mapping[obs] for obs in obs]

        for iter_no in range(n_iterations):

            fwd_probs = self.forward(obs)
            bwd_probs = self.backward(obs)

            seq_prob = np.sum(fwd_probs[-1])
            T = len(obs)

            gamma = np.zeros((T, self.n_states))

            for t in range(T):
                gamma[t] = (fwd_probs[t] * bwd_probs[t]) / seq_prob
            
            new_emission = np.zeros((T, self.n_states, self.n_states))
            
            for t in range(T-1):
                for i in range(self.n_states):  # current day state
                    for j in range(self.n_states):  # next day state

                        prob = fwd_probs[t, i] * self.transition_probs[i, j] * self.emission_probs[j, obs[t+1]] * bwd_probs[t+1, j]
                        new_emission[t, i, j] = prob
                
                new_emission[t] /= np.sum(new_emission[t])
            
            # Updating values

            # initial probs
            self.initial_probs = gamma[0]

            # transition probs
            for i in range(self.n_states):
                for j in range(self.n_states):

                    prob = np.sum(new_emission[:, i, j]) / np.sum(gamma[:-1, i])
                    self.transition_probs[i, j] = prob
            
            # emission probs
            for i in range(self.n_states):
                for j in range(self.n_observations):

                    J_obs_mask = (np.array(obs) == j)
                    prob = np.sum(gamma[J_obs_mask, i]) / np.sum(gamma[:, i])
                    self.emission_probs[i, j] = prob
            
            self.transition_probs = (self.transition_probs / np.sum(self.transition_probs, axis=-1, keepdims=True))
            self.emission_probs = (self.emission_probs / np.sum(self.emission_probs, axis=-1, keepdims=True))


In [ ]:
hmm = HiddenMarkovModel(init_probs, trans_probs, emit_probs)

In [ ]:
hmm.baum_welch(obs, n_iterations=50)

In [ ]:
hmm.emission_probs

In [ ]:
fwd_matrix = hmm.backward(obs)